<a href="https://colab.research.google.com/github/josepeon/python_dad_class/blob/main/trainers_pretrained_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [87]:
import pandas as pd
import numpy as np

In [88]:
from datasets import load_dataset
ds = load_dataset('jackhhao/jailbreak-classification')

In [89]:
ds

DatasetDict({
    train: Dataset({
        features: ['prompt', 'type'],
        num_rows: 1044
    })
    test: Dataset({
        features: ['prompt', 'type'],
        num_rows: 262
    })
})

In [90]:
ds['train'][0]

{'prompt': 'You are a devoted fan of a celebrity.', 'type': 'benign'}

In [91]:
ds['train'][1]

{'prompt': 'You are Joseph Seed from Far Cry 5. Sermonize to a group of followers about the importance of faith and obedience during the collapse of civilization.',
 'type': 'benign'}

In [92]:
from transformers import BertForSequenceClassification
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
#tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
#model = BertModel.from_pretrained("bert-base-uncased")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [93]:
#tokenizer(ds['train'][0]['prompt'])

In [94]:
def encode(example):
    return tokenizer(example['prompt'], truncation=True, padding=True, max_length=128)

In [95]:
data = ds.map(encode)

In [96]:
ds['train']

Dataset({
    features: ['prompt', 'type'],
    num_rows: 1044
})

In [97]:
def targeter(examples):
  return {'type': 1 if examples['type'] == 'jailbreak' else 0}

In [98]:
ds.map(targeter)

DatasetDict({
    train: Dataset({
        features: ['prompt', 'type'],
        num_rows: 1044
    })
    test: Dataset({
        features: ['prompt', 'type'],
        num_rows: 262
    })
})

In [99]:
data = ds.map(encode)

In [100]:
data

DatasetDict({
    train: Dataset({
        features: ['prompt', 'type', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1044
    })
    test: Dataset({
        features: ['prompt', 'type', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 262
    })
})

In [101]:
data['train'][0]

{'prompt': 'You are a devoted fan of a celebrity.',
 'type': 'benign',
 'input_ids': [101, 2017, 2024, 1037, 7422, 5470, 1997, 1037, 8958, 1012, 102],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [102]:
from transformers import Trainer, TrainingArguments

In [103]:
ta = TrainingArguments('testing-jailbreak', remove_unused_columns=True)

In [104]:
from datasets import ClassLabel

#Get unique labels
unique_labels = list(set(data['train']['type']))
class_labels = ClassLabel(names=unique_labels)

#Map string labels to integers
def encode_labels(example):
  example['labels'] = class_labels.str2int(example['type'])
  return example

data = data.map(encode_labels)

In [105]:
trainer = Trainer(model = model,
                  args = ta,
                  train_dataset = data['train'],
                  eval_dataset = data['test'],
                  tokenizer = tokenizer)

/tmp/ipython-input-2083910605.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model = model,


In [106]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=393, training_loss=0.08428407625387643, metrics={'train_runtime': 21.6112, 'train_samples_per_second': 144.925, 'train_steps_per_second': 18.185, 'total_flos': 205806289724640.0, 'train_loss': 0.08428407625387643, 'epoch': 3.0})